# Hindu Numeral Recognition

CNN classifier for 32x32 grayscale Devanagari digits 0-9. This commit covers data loading, exploration, train/validation split and the data generators.

# Disclaimer
AI tools were used for code completion and Academic writing

## 1. Imports

Standard data-science stack. Random seed fixed across Python, NumPy and TensorFlow so the train/validation split and weight initialisation are reproducible.

*Student 2: added tensorflow / keras imports for the model.*

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Configuration

All paths and hyperparameters in one place. Images are already 32x32 grayscale, so no resizing is required.

*Student 2: added `EPOCHS` and `MODEL_PATH`.*

In [ ]:
BASE_DIR    = Path('iivp-2026-challenge')
TRAIN_DIR   = BASE_DIR / 'train' / 'train'
TRAIN_CSV   = BASE_DIR / 'train.csv'

IMG_SIZE    = 32
CHANNELS    = 1
BATCH_SIZE  = 128
EPOCHS      = 30
NUM_CLASSES = 10
MODEL_PATH  = 'best_model.keras'

## 3. Training labels

`train.csv` lists `Id` and `Category` for every training image. A `filepath` column is built pointing to `train/train/{Category}/{Id}.png` so the data generator can stream the files directly.

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df['filepath'] = df.apply(
    lambda r: str(TRAIN_DIR / str(r['Category']) / f"{r['Id']}.png"), axis=1
)
df['Category'] = df['Category'].astype(str)

print(f"Total training samples: {len(df)}")
print("\nClass distribution:")
print(df['Category'].value_counts().sort_index())

17,000 samples, exactly 1,700 per class. Perfectly balanced - no class weighting needed.

## 4. Sample inspection

One example per class to confirm the files load, are grayscale, and to get a feel for the digit shapes (Devanagari numerals look very different from Western 0-9).

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit in range(10):
    sample = df[df['Category'] == str(digit)].iloc[0]
    img = Image.open(sample['filepath'])
    ax = axes[digit // 5][digit % 5]
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Digit: {digit}')
    ax.axis('off')
plt.suptitle('One sample per class', fontsize=14)
plt.tight_layout()
plt.show()

sample_arr = np.array(Image.open(df.iloc[0]['filepath']))
print(f"Image shape: {sample_arr.shape}, dtype: {sample_arr.dtype}, range: [{sample_arr.min()}, {sample_arr.max()}]")

Pixel values are uint8 in [0, 255]. The generator below rescales them to [0, 1].

## 5. Train / validation split

10% (1,700 images) held out for validation. Stratified by class so each digit ends up with exactly 170 validation images.

In [ ]:
train_df, val_df = train_test_split(
    df, test_size=0.10, stratify=df['Category'], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")
print("Val class distribution:")
print(val_df['Category'].value_counts().sort_index())

170 per class in val, as expected.

## 6. Data generators

**Training generator:** rescale to [0, 1] plus light augmentation - small rotations, shifts and zooms - to simulate handwriting variation. No flipping: a horizontally flipped Devanagari digit is a different character.

**Validation generator:** rescaling only, no augmentation, for a clean accuracy measurement.

`color_mode='grayscale'` is required - the default is RGB, which would silently load 3-channel images and break the model input shape.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=False,
    vertical_flip=False,
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='Category',
    target_size=(IMG_SIZE, IMG_SIZE), color_mode='grayscale',
    class_mode='sparse', batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)
val_gen = val_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='Category',
    target_size=(IMG_SIZE, IMG_SIZE), color_mode='grayscale',
    class_mode='sparse', batch_size=BATCH_SIZE, shuffle=False
)

## 7. Batch sanity check

Pull one batch from `train_gen` and confirm the shape is `(128, 32, 32, 1)` and pixel values fall in [0, 1].

In [ ]:
x_batch, y_batch = next(train_gen)
print(f"Batch shape: {x_batch.shape}")
print(f"Pixel range after rescaling: [{x_batch.min():.3f}, {x_batch.max():.3f}]")
print(f"Label sample: {y_batch[:10]}")

train_gen.reset()

Data side complete: dataframe loaded, stratified split, augmented generators, batch shape verified.

---

##Model architecture and training

## 8. CNN architecture

Two convolutional blocks (Conv-BN-ReLU x2 -> MaxPool -> Dropout) followed by a Dense(128) head and softmax over 10 classes.

- Two blocks: spatial size goes 32 -> 16 -> 8. A third block was tried and dropped - 4x4 feature maps were too coarse and validation accuracy fell.
- Filters grow 32 -> 64: deeper layers combine simple edges into more complex shape features and need more capacity.
- BatchNorm after every Conv: stabilises training, faster convergence.
- Dropout 0.25 / 0.25 / 0.5: regularisation against memorising the 17k-sample training set.

In [ ]:
def build_model():
    model = keras.Sequential([
        # Block 1
        layers.Conv2D(32, 3, padding='same', input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(32, 3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),
        # Block 2
        layers.Conv2D(64, 3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, 3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.25),
        # Head
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    return model

model = build_model()
model.summary()

Around 591k parameters (~2.3 MB). Small enough to train on CPU.

## 9. Compile and callbacks

- Adam, lr=1e-3 - safe default.
- `sparse_categorical_crossentropy` - integer labels paired with softmax output.
- Accuracy as the reporting metric (matches the competition criterion).

Callbacks:
- `ModelCheckpoint` - persists the weights whenever validation accuracy improves.
- `EarlyStopping` - halts training after 5 epochs of no val-loss improvement; restores best weights.
- `ReduceLROnPlateau` - halves the learning rate after 3 epochs of no improvement, down to 1e-6.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

## 10. Training

Up to 30 epochs. Each epoch processes the 15,300 augmented training images and evaluates on the 1,700 validation images. Early stopping typically halts the run before epoch 30.

In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 11. Training curves

Train vs validation accuracy and loss across epochs. A small gap between the two curves indicates the model is generalising rather than memorising.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()

best_val_acc = max(history.history['val_accuracy'])
print(f"Best validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")

Best weights saved to `best_model.keras`. Ready for evaluation and test-set predictions.